In [1]:
from utils import get_spark_session, cargar_tabla, guardar_tabla, udf_limpieza
from pyspark.sql.functions import col, ceil, dayofmonth, month, year, date_format

spark = get_spark_session()

25/04/30 23:50:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/30 23:50:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/30 23:50:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/04/30 23:50:23 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


### Cargar las tablas

In [11]:
clientes = cargar_tabla("clientes", spark)
productos = cargar_tabla("productos", spark)
corresponsales = cargar_tabla("corresponsales", spark)
pagos = cargar_tabla("pagos", spark)
ordenes = cargar_tabla("ordenes_compra", spark)
detalle = cargar_tabla("detalle_orden", spark)

### Limpieza de dimensiones

In [12]:
dim_cliente = clientes.select("id_cliente", udf_limpieza("nombre").alias("nombre"), udf_limpieza("apellido").alias("apellido"))
dim_producto = productos.select("id_producto", udf_limpieza("nombre").alias("nombre"), (ceil(col("precio")*100)/100.0).alias("precio"))
dim_corresponsal = corresponsales.select("id_corresponsal", udf_limpieza("nombre").alias("nombre"))
dim_tiempo = pagos.select(col("fecha_pago").alias("fecha")).distinct()\
    .withColumn("dia", dayofmonth("fecha"))\
    .withColumn("mes", month("fecha"))\
    .withColumn("anio", year("fecha"))\
    .withColumn("mes_anio", date_format("fecha", "yyyy-MM"))

### Integración con tablas de hechos

In [13]:
df_join1 = ordenes.join(detalle, "id_orden")
df_hechos = df_join1.join(pagos, "id_orden")\
    .select("id_orden", "id_cliente", "id_producto", "id_corresponsal",
            col("fecha_pago").alias("fecha"), "cantidad", 
            (ceil(col("precio_unitario")*100)/100.0).alias("precio_unitario"), 
            (ceil(col("precio_unitario")*col("cantidad")*100)/100.0).alias("monto_total"))


### Guardar tablas

In [14]:
guardar_tabla(dim_cliente, "dim_cliente")
guardar_tabla(dim_producto, "dim_producto")
guardar_tabla(dim_corresponsal, "dim_corresponsal")
guardar_tabla(dim_tiempo, "dim_tiempo")
guardar_tabla(df_hechos, "fac_ventas")
